# Pra-pemrosesan Data (Text Preprocessing)

Tahap ini merupakan persiapan krusial dalam pemrosesan Natural Language Processing (NLP). Kita akan membersihkan ulasan dari _noise_ dan mengubah teks ke bentuk dasar, serta melakukan Label Encoding untuk variabel target sentimen.

In [1]:
import pandas as pd
import re
import emoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.preprocessing import LabelEncoder

# Load dataset hasil EDA
df = pd.read_csv("../data/processed/dataset_fitur.csv")
print(f"Jumlah data awal: {len(df)}")

Jumlah data awal: 673


## 1. Menangani Missing Values & Outliers

Sesuai temuan EDA, beberapa ulasan mungkin kosong atau sangat panjang (outlier > 150 kata).

In [2]:
# Hapus ulasan kosong (jika ada)
df.dropna(subset=['ulasan'], inplace=True)

# Hapus outlier ekstrim (misal: spam atau cerita bertele-tele yang mengacaukan model)
sebelum = len(df)
df = df[df['jumlah_kata'] <= 150].copy()
print(f"Dihapus {sebelum - len(df)} baris ulasan (outliers).")
print(f"Sisa data: {len(df)}")

Dihapus 7 baris ulasan (outliers).
Sisa data: 666


## 2. Inisialisasi Pustaka (Sastrawi)
Kita menyiapkan _stemmer_ (mengubah imbuhan ke kata dasar) dan _stopwords_ (membuang kata hubung tidak penting). Penting untuk mempertahankan kata negasi agar sentimen tidak meleset!

In [3]:
stemmer = StemmerFactory().create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopwords = stopword_factory.get_stop_words()

# KATA NEGASI JANGAN DIHAPUS!
negasi = ['tidak', 'kurang', 'belum', 'jangan', 'bukan', 'tak']
stopwords = [word for word in stopwords if word not in negasi]

## 3. Fungsi Preprocessing (Cleansing, Emoji, Stemming)

In [4]:
def preprocess_text(text):
    try:
        text = str(text).lower() # Case folding
        text = emoji.demojize(text, language="id") # Konversi emoji 😭 -> wajah_menangis
        text = text.replace(':', ' ').replace('_', ' ')
        
        text = re.sub(r'@[A-Za-z0-9_]+', '', text) # Hapus mention
        text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Hapus URL
        text = re.sub(r'[^a-z\s]', ' ', text) # Hapus tanda baca/angka
        text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi ganda
        
        # Stopwords
        words = [w for w in text.split() if w not in stopwords]
        
        # Stemming
        return stemmer.stem(" ".join(words))
    except Exception as e:
        return ""


## 4. Eksekusi Preprocessing & Feature Encoding
Karena Sastrawi cukup berat, proses ini bisa memakan waktu sekitar 1-2 menit.

In [5]:
# Jalankan preprocessing (memakan waktu ~1 menit)
df['ulasan_bersih'] = df['ulasan'].apply(preprocess_text)

# Hapus baris yang kosong akibat pembersihan (misal cuma isi tanda seru saja)
df = df[df['ulasan_bersih'].str.strip() != '']

# Encoding label: (Negatif -> 0, Netral -> 1, Positif -> 2)
encoder = LabelEncoder()
mapping_sentimen = {"Negatif": 0, "Netral": 1, "Positif": 2}
df['label_encoded'] = df['label_sentimen'].map(mapping_sentimen)

# Tampilkan perbandingan sebelum dan sesudah
df[['ulasan', 'ulasan_bersih', 'label_encoded']].head()

,ulasan,ulasan_bersih,label_encoded
0,"Produk: Barang rusak | Sorry to say, perdana g...",produk barang rusak sorry to say perdana gue b...,0
1,pertama kali make forebie beruntusan&jerawat m...,pertama kali make forebie beruntus jerawat mul...,0
2,"Mungkin setiap org cok""kan,di aku ga ccok pana...",mungkin org cok kan aku ga ccok panas gatel je...,0
3,kaget banget baru semingguan pakai langsung bi...,kaget banget baru minggu pakai langsung bintik...,0
4,"bekas jerawat pie malah jadi bopeng 😢, bopeng ...",bekas jerawat pie malah jadi bopeng wajah mena...,0


## 5. Simpan Hasil Bersih
Menyimpan ke file baru `dataset_cleaned.csv` agar tidak perlu mengulang *stemming* lagi saat proses Modeling ML.

In [6]:
df.to_csv("../data/processed/dataset_cleaned.csv", index=False, encoding="utf-8")
print(f"Data bersih tersimpan! Total baris siap modeling: {len(df)}")

Data bersih tersimpan! Total baris siap modeling: 666
